# Exercise 13 - Object Detection

Object Detection using Single Shot Detection (SSD) and You Only Look Once (YOLO) Model

In this lab we work with a COCO dataset pretrained model for object detection. The model uses MobileNet as a base and thus it is referred to as MobileNet Single Shot Detection(SSD) model. Like previous notebooks, we present some examples of object detection model (ssd_mobilenet_v1.tflite).

- Code to read saved images and perform object detection (Step 1)
- Check if object of interest is within a certain interest and detected with expected confidence (Step 2)
- Extend code to perform object detection of pictures taken by your camera (optional).  The image is stored as photo.jpg and later called by model from step 1.

- Run this exercise using **CPU** or T4 GPU.  We carry out model inferencing in this exercise.

**Step 0:**

- Upload zip file containing model and sample image files.
- Install LiteRT (previously known as TensorFlow Lite) runtime.
- Create a labels file called coco_labels.txt to map index to text description .

In [ ]:
%%bash

wget -q https://pdl-doulos.s3.us-west-2.amazonaws.com/Object-Detection.zip
unzip -q Object-Detection.zip

In [ ]:
!pip install -q ai-edge-litert

In [ ]:
%%writefile coco_labels.txt
person
bicycle
car
motorcycle
airplane
bus
train
truck
boat
traffic light
fire hydrant
street sign
stop sign
parking meter
bench
bird
cat
dog
horse
sheep
cow
elephant
bear
zebra
giraffe
hat
backpack
umbrella
shoe
eye glasses
handbag
tie
suitcase
frisbee
skis
snowboard
sports ball
kite
baseball bat
baseball glove
skateboard
surfboard
tennis racket
bottle
plate
wine glass
cup
fork
knife
spoon
bowl
banana
apple
sandwich
orange
broccoli
carrot
hot dog
pizza
donut
cake
chair
couch
potted plant
bed
mirror
dining table
window
desk
toilet
door
tv
laptop
mouse
remote
keyboard
cell phone
microwave
oven
toaster
sink
refrigerator
blender
book
clock
vase
scissors
teddy bear
hair drier
toothbrush
hair brush

**Step 1:** Inference code for object detection  

 Ninety labels from COCO dataset are part of inference code.  Multiple images are refernced in lines 95 to 97 for inference. Experiment with different images. You can even upload your own image.

Output image with bounding boxes is saved as *object-detected.jpg*.
You can find and view the output image using file explorer icon located in top left panel column.

The bounding box locations are presented in the output tensor as a multidimensional array of [N][4] floating point values between 0 and 1; the inner arrays representing bounding boxes in the form [top, left, bottom, right]

In [ ]:
import os
import time
import cv2
from google.colab.patches import cv2_imshow
from ai_edge_litert.interpreter import Interpreter
import numpy as np

BASE_DIR = os.getcwd()
LABELS_FILE = os.path.join(BASE_DIR, 'coco_labels.txt')
DOG_IMAGE = os.path.join(BASE_DIR, 'Object-Detection', 'Images', 'dog.jpg')
APPLE_IMAGE = os.path.join(BASE_DIR, 'Object-Detection', 'Images', 'apple.jpeg')
CAR_IMAGE = os.path.join(BASE_DIR, 'Object-Detection', 'Images', 'car-road.jpg')
SSD_MOBILENET_MODEL = os.path.join(BASE_DIR, 'Object-Detection', 'Model-Files', 'ssd_mobilenet_v1.tflite')
IMAGE_BBOX_FILE = os.path.join(BASE_DIR, 'object-detected.jpg')

# Function to load labels from a file
def load_labels(file_path):
    with open(file_path, 'r') as f:
        return {i: line.strip() for i, line in enumerate(f.readlines())}

# Load labels from the provided text file
label2string = load_labels(LABELS_FILE)

def detect_from_image():
    # prepare input image
    start = time.time()
    #img_org = cv2.imread(DOG_IMAGE)
    img_org = cv2.imread(APPLE_IMAGE)
    #img_org = cv2.imread('/content/Object-Detection/photo.jpg')
    cv2_imshow(img_org)
    img = cv2.cvtColor(img_org, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (300, 300))
    img = img.reshape(1, img.shape[0], img.shape[1],
                      img.shape[2])  # (1, 300, 300, 3)
    img = img.astype(np.uint8)

    # Overview of Object Detection: https://www.tensorflow.org/lite/examples/object_detection/overview
    # Load pretrained model:https://tfhub.dev/tensorflow/lite-model/ssd_mobilenet_v1/1/default/1

    interpreter = Interpreter(
        model_path=SSD_MOBILENET_MODEL)

    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    # set input tensor
    interpreter.set_tensor(input_details[0]['index'], img)

    # run
    interpreter.invoke()

    # get output tensor
    boxes = interpreter.get_tensor(output_details[0]['index'])
    boxes_shape = output_details[0]['shape_signature']
    labels = interpreter.get_tensor(output_details[1]['index'])
    scores = interpreter.get_tensor(output_details[2]['index'])
    num = interpreter.get_tensor(output_details[3]['index'])
    labels_list = labels.tolist()

    # Convert boxes to a NumPy array with a suitable data type
    boxes = np.array(boxes, dtype=np.float32)

    print('Bounding Box coordinates:', boxes)
    print('Boxes_shape:', boxes_shape)

    print (labels_list)

    for i in range (5): # Top 5 detections
      print('Label:', label2string[labels[0][i]], ',Score:', scores[0][i], ',Bounding box coodinates:', boxes [0][i])

    for i in range(boxes.shape[1]):
        if scores[0, i] > 0.50:
            box = boxes[0, i, :]
            x0 = int(box[1] * img_org.shape[1])
            y0 = int(box[0] * img_org.shape[0])
            x1 = int(box[3] * img_org.shape[1])
            y1 = int(box[2] * img_org.shape[0])
            box = box.astype(np.uint8)
            cv2.rectangle(img_org, (x0, y0), (x1, y1), (255, 0, 0), 2)
            #cv2.rectangle(img_org, (x0, y0), (x0 + 100, y0 - 30), (255, 0, 0), -1)
            cv2.putText(img_org, str(label2string[labels[0][i]]), (x0, y0),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

    cv2.imwrite(IMAGE_BBOX_FILE, img_org)
    stop = time.time()
    print(f'time for inference is {stop-start:.2f} seconds')
    processed_image = cv2.imread(IMAGE_BBOX_FILE)
    cv2_imshow(processed_image)

if __name__ == '__main__':
    detect_from_image()

**Step 2:** Image analytics using object detection.

- Use Image *car-road.jpg* for finding location of car within an image.
- Understand code that checks if the car is within a specified region shown as green box.
- The output image with bounding boxes is saved as object-detected.jpg

You can find and view the output image using file explorer icon located in top left panel column.

In [ ]:
import time
import cv2
from google.colab.patches import cv2_imshow
from ai_edge_litert.interpreter import Interpreter
import numpy as np

# Function to load labels from a file
def load_labels(file_path):
    with open(file_path, 'r') as f:
        return {i: line.strip() for i, line in enumerate(f.readlines())}

# Load labels from the provided text file
label2string = load_labels(LABELS_FILE)


def detect_from_image():
    # prepare input image
    start = time.time()
    img_org = cv2.imread(CAR_IMAGE)
    cv2_imshow(img_org)
    img = cv2.cvtColor(img_org, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (300, 300))
    img = img.reshape(1, img.shape[0], img.shape[1],
                      img.shape[2])  # (1, 300, 300, 3)
    img = img.astype(np.uint8)

    # Overview of Object Detection: https://www.tensorflow.org/lite/examples/object_detection/overview
    # Load pretrained model:https://tfhub.dev/tensorflow/lite-model/ssd_mobilenet_v1/1/default/1

    interpreter = Interpreter(
        model_path= SSD_MOBILENET_MODEL)

    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    # set input tensor
    interpreter.set_tensor(input_details[0]['index'], img)

    # run
    interpreter.invoke()

    # get output tensor

    boxes = interpreter.get_tensor(output_details[0]['index'])
    boxes_shape = output_details[0]['shape_signature']
    labels = interpreter.get_tensor(output_details[1]['index'])
    scores = interpreter.get_tensor(output_details[2]['index'])
    num = interpreter.get_tensor(output_details[3]['index'])
    labels_list = labels.tolist()

    # Convert boxes to a NumPy array with a suitable data type
    boxes = np.array(boxes, dtype=np.float32)

    print('Boxes:', boxes)
    print('Boxes_shape:', boxes_shape)

    #draw expected box for expected location of car
    x0 = int(0.42 * img_org.shape[1])
    y0 = int(0.40 * img_org.shape[0])
    x1 = int(0.70 * img_org.shape[1])
    y1 = int(0.60 * img_org.shape[0])
    #box = box.astype(np.int)
    cv2.rectangle(img_org, (x0, y0), (x1, y1), (0, 255, 0), 2)
    #cv2.rectangle(img_org, (x0, y0), (x0 + 100, y0 - 30), (255, 0, 0), -1)
    cv2.putText(img_org, 'car expected', (x0, y0),  cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)


    for i in range(boxes.shape[1]):
        if scores[0, i] > 0.50:
            box = boxes[0, i, :]
            x0 = int(box[1] * img_org.shape[1])
            y0 = int(box[0] * img_org.shape[0])
            x1 = int(box[3] * img_org.shape[1])
            y1 = int(box[2] * img_org.shape[0])
            box = box.astype(np.uint8)
            cv2.rectangle(img_org, (x0, y0), (x1, y1), (255, 0, 0), 2)
            #cv2.rectangle(img_org, (x0, y0), (x0 + 100, y0 - 30), (255, 0, 0), -1)
            cv2.putText(img_org, str(label2string[labels[0][i]]), (x0, y0),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

    cv2.imwrite('object-detected-car.jpg', img_org)
    stop = time.time()
    print(f'time for inference is {stop-start:.2f} seconds')
    processed_image = cv2.imread('object-detected-car.jpg')
    cv2_imshow(processed_image)

# Code to detect if object car is detected with specified level of confidence and
# is in the right area of the intersection

    for i in range(labels.shape[1]):
      if ((scores[0,i] > 0.5) and (np.array(labels[0,i] == 2.0)) and (boxes[0,i,0]>= 0.40) and (boxes[0,i,1]>= 0.42)
          and (boxes[0,i,2] <= 0.60) and (boxes[0,i,3]<= 0.70)):
        print (f'{label2string[labels[0][i]]} in right area')
        print (f'box details {boxes[0,i,:]}')
      else:
        pass


if __name__ == '__main__':
    detect_from_image()

**Exercise: 1**



*   Modify code in step 1 to use image dog.jpg
*   Using bounding box co-ordinates from inference output,determine if the dog is sitting on the bowl.

### Solution to Exercise 1


*   Use object detector model on image - dog.jpg
*   Using bounding box co-ordinates from inference output,determine if the dog is sitting on the bowl.

- The bounding box locations are presented in the output tensor as a multidimensional array of [N][4] floating point values between 0 and 1; the inner arrays representing bounding boxes in the form [top, left, bottom, right]

- If the bounding box containing the dog, overlaps with the box containing the bowl, the dog is probably sitting on the bowl.

In [ ]:
import time
import cv2
from ai_edge_litert.interpreter import Interpreter
import numpy as np


# Function to load labels from a file
def load_labels(file_path):
    with open(file_path, 'r') as f:
        return {i: line.strip() for i, line in enumerate(f.readlines())}

# Load labels from the provided text file
label2string = load_labels(LABELS_FILE)


def detect_from_image():
    # prepare input image
    start = time.time()
    img_org = cv2.imread(DOG_IMAGE)
    #img_org = cv2.imread(APPLE_IMAGE)
    #img_org = cv2.imread('/content/Object-Detection/photo.jpg')
    #	cv2.imshow('image', img)
    img = cv2.cvtColor(img_org, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (300, 300))
    img = img.reshape(1, img.shape[0], img.shape[1],
                      img.shape[2])  # (1, 300, 300, 3)
    img = img.astype(np.uint8)

    # Overview of Object Detection: https://www.tensorflow.org/lite/examples/object_detection/overview
    # Load pretrained model:https://tfhub.dev/tensorflow/lite-model/ssd_mobilenet_v1/1/default/1

    interpreter = Interpreter(
        model_path=SSD_MOBILENET_MODEL)

    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    # set input tensor
    interpreter.set_tensor(input_details[0]['index'], img)

    # run
    interpreter.invoke()

    # get output tensor
    boxes = interpreter.get_tensor(output_details[0]['index'])
    boxes_shape = output_details[0]['shape_signature']
    labels = interpreter.get_tensor(output_details[1]['index'])
    scores = interpreter.get_tensor(output_details[2]['index'])
    num = interpreter.get_tensor(output_details[3]['index'])
    labels_list = labels.tolist()

    # Convert boxes to a NumPy array with a suitable data type
    boxes = np.array(boxes, dtype=np.float32)
    print('Bounding Box coordinates:', boxes)

    print('Boxes_shape:', boxes_shape)
    print (labels_list)

    for i in range (5): # Top 5 detections
      print('Label:', label2string[labels[0][i]], ',Score:', scores[0][i], ',Bounding box coodinates:', boxes [0][i])

    for i in range(boxes.shape[1]):
        if scores[0, i] > 0.60:
            box = boxes[0, i, :]
            x0 = int(box[1] * img_org.shape[1])
            y0 = int(box[0] * img_org.shape[0])
            x1 = int(box[3] * img_org.shape[1])
            y1 = int(box[2] * img_org.shape[0])
            box = box.astype(np.uint8)
            cv2.rectangle(img_org, (x0, y0), (x1, y1), (255, 0, 0), 2)
            #cv2.rectangle(img_org, (x0, y0), (x0 + 100, y0 - 30), (255, 0, 0), -1)
            cv2.putText(img_org, str(label2string[labels[0][i]]), (x0, y0),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

    cv2.imwrite('object-detected.jpg', img_org)
    stop = time.time()
    print(f'time for inference is {stop-start:.2f} seconds')
    processed_image = cv2.imread('object-detected.jpg')
    cv2_imshow(processed_image)

#Code to check for lower confidence on detection

if __name__ == '__main__':
    detect_from_image()